# Lab 01. 데이터 품질과 전처리

전처리의 목표는 결측을 0개로 만드는 것이 아니라 분석 목적에 맞는 품질 기준을 세우고 처리 영향을 기록하는 것이다.
이 Notebook에서는 학생별로 다른 결측, 중복과 이상값을 가진 데이터로 연습한다.

IQR 이상값 후보 경계:

$$IQR=Q_3-Q_1,\qquad [Q_1-1.5IQR,\;Q_3+1.5IQR]$$

In [ ]:
from pathlib import Path
import sys, subprocess

REPO_URL = "https://github.com/niko2204/bigdataservice.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/bigdataservice")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p.resolve() for p in candidates if (p / "src").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("bigdataservice 저장소 루트에서 Notebook을 실행하세요.")

sys.path.insert(0, str(ROOT))
STUDENT_ID = "20260001"  # 반드시 본인 학번으로 변경
print("저장소:", ROOT)
print("실습 학번:", STUDENT_ID)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.education.personalized_data import make_student_dataset

raw = make_student_dataset(STUDENT_ID)
print("원본 크기:", raw.shape)
display(raw.head())

## 1. 완성 예제: 품질 프로파일

자료형, 결측 수·비율, 고유값 수를 한 표로 만든다. 완전 중복과 특정 식별자 중복은 구분해야 한다.

In [ ]:
quality = pd.DataFrame({
    "자료형": raw.dtypes.astype(str),
    "결측수": raw.isna().sum(),
    "결측률": raw.isna().mean().round(3),
    "고유값수": raw.nunique(dropna=False),
})
print("완전 중복 행:", raw.duplicated().sum())
display(quality)

## 2. 완성 예제: 결측 대체 방법 비교

전체 중앙값과 업종별 중앙값을 비교한다. 평균 대체도 가능하지만 이상값에 민감하고 분산을 줄일 수 있다.

In [ ]:
compare = raw.copy()
compare["임대료_전체중앙값"] = compare["월임대료"].fillna(compare["월임대료"].median())
compare["임대료_업종중앙값"] = compare["월임대료"].fillna(
    compare.groupby("업종")["월임대료"].transform("median")
)
rows = raw["월임대료"].isna()
display(compare.loc[rows, ["업종", "월임대료", "임대료_전체중앙값", "임대료_업종중앙값"]])

## 3. 완성 예제: IQR 이상값은 후보

경계 밖 행을 바로 삭제하지 않고 업종, 지역과 다른 변수를 함께 확인한다.

In [ ]:
sales = raw["월매출"].dropna()
q1, q3 = sales.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outlier_mask = ~raw["월매출"].between(lower, upper)
print(f"Q1={q1:.1f}, Q3={q3:.1f}, IQR={iqr:.1f}, 경계=({lower:.1f}, {upper:.1f})")
display(raw.loc[outlier_mask].sort_values("월매출", ascending=False))

## 4. 따라하기: 정제 파이프라인

아래 정책은 하나의 예일 뿐이다. 유동인구와 임대료는 업종별 중앙값으로 대체하고 완전 중복을 제거한다.
이상값은 분석용 플래그만 만들고 원자료 값은 유지한다.

In [ ]:
clean = raw.drop_duplicates().copy()
for column in ["유동인구", "월임대료"]:
    clean[column] = clean[column].fillna(
        clean.groupby("업종")[column].transform("median")
    )
clean["매출_이상후보"] = ~clean["월매출"].between(lower, upper)

assert clean.duplicated().sum() == 0
assert clean[["유동인구", "월임대료"]].isna().sum().sum() == 0
print("정제 크기:", clean.shape)
display(clean["매출_이상후보"].value_counts())

## 5. 처리 전후 영향 비교

결측 대체와 중복 제거가 평균, 중앙값과 표준편차에 미친 영향을 확인한다.

In [ ]:
before = raw["월임대료"].agg(["count", "mean", "median", "std"])
after = clean["월임대료"].agg(["count", "mean", "median", "std"])
comparison = pd.concat([before.rename("처리전"), after.rename("처리후")], axis=1)
display(comparison)

## 6. 독립 연습

1. 유동인구 결측을 전체 평균, 전체 중앙값, 업종별 중앙값으로 각각 대체한다.
2. 세 방법의 평균·표준편차와 유동인구–월매출 상관계수를 비교한다.
3. IQR 계수를 1.5와 3.0으로 바꾸어 이상 후보 수를 비교한다.
4. 최종 처리 정책을 선택하고 데이터 생성 과정과 분석 목적을 근거로 5문장 이상 설명한다.

In [ ]:
# TODO: 세 대체 방법 비교표 작성
imputation_comparison = None
display(imputation_comparison)

# TODO: IQR 1.5와 3.0의 이상 후보 행 번호와 개수 비교
outlier_comparison = None
display(outlier_comparison)

## 7. 자가점검

- [ ] 원본 DataFrame을 변경하지 않았다.
- [ ] 결측·중복·이상값을 서로 다른 문제로 처리했다.
- [ ] 이상값을 삭제하기 전에 원인과 영향을 확인했다.
- [ ] 처리 전후 행 수와 통계량을 기록했다.
- [ ] 선택하지 않은 대안의 결과도 비교했다.